# A/B Testing & Statistical Significance Analysis

Analysis of the Marketing A/B Testing dataset. The notebook imports shared functions from `ab_test_analysis.py` so script and notebook calculations stay consistent.

**Interpretation:** causal conclusions depend on assignment and measurement assumptions. The dataset does not document the intended allocation ratio or include a genuine pre-treatment CUPED covariate. Run all cells from the repository root after installing `requirements.txt`.

In [ ]:
from ab_test_analysis import (
    load_data, check_srm, run_ztest, bootstrap_validation,
    business_impact, power_analysis, synthetic_cuped_demo, regression_and_hte,
)

df = load_data()
df.head()

## 1. Validation and SRM

The loader validates required columns, binary outcomes, group labels, missing values, ad counts, and duplicate user IDs. Because the intended allocation ratio is not documented, SRM is **not tested** by default. Supply a ratio only when supported by the experiment specification.

In [ ]:
srm_result = check_srm(df)
# Only if the experiment specification documents the intended split:
# srm_result = check_srm(df, expected_ratio=(0.96, 0.04))

## 2. Primary conversion comparison

The two-proportion Z-test reports conversion rates, absolute and relative lift, p-value, and an approximate 95% confidence interval. Statistical significance does not by itself establish business value or causality.

In [ ]:
ztest_result = run_ztest(df)

## 3. Bootstrap validation

The fixed seed makes the percentile interval reproducible. For binary outcomes, binomial draws are mathematically equivalent to resampling binary observations with replacement.

In [ ]:
bootstrap_result = bootstrap_validation(df, n_boot=10_000, seed=42)

## 4. Illustrative business impact

This compares observed ad conversions with the number expected at the PSA conversion rate. It is conditional on valid randomization and measurement—not proof that the difference was caused by ads. Cost and guardrail metrics are unavailable.

In [ ]:
impact_result = business_impact(df)

## 5. Retrospective MDE

The MDE describes the approximate effect detectable with the observed sample sizes at the specified alpha and target power. This is retrospective and not independent proof that the experiment was well-powered.

In [ ]:
mde_result = power_analysis(df)

## 6. CUPED — separate synthetic demonstration

The marketing data has no genuine pre-treatment metric. To avoid outcome leakage, this demo creates a separate synthetic randomized experiment: a pre-treatment covariate is generated first, then outcomes are simulated from that covariate and treatment assignment. These results do **not** describe the marketing dataset.

In [ ]:
cuped_result = synthetic_cuped_demo(n=20_000, seed=42)

## 7. Exploratory regression and HTE

These models are retained for exploration. `total_ads` is a post-treatment exposure variable, so adjusting for it or interacting it with treatment does not provide a clean causal estimate of effect heterogeneity. A nonsignificant interaction is not evidence that effects are equal across exposure levels.

In [ ]:
regression_results = regression_and_hte(df)